# Run All Experiments (Notebook)

**Workflow:**
1. Run "Imports" and "Configuration" cells once
2. Run "Load Data" cell once — first time is slow (HF + tokenization), subsequent runs hit a pickle cache at `data/prepared/*.pkl` and are near-instant even after a kernel restart
3. Edit strategy/experiment config and re-run "Run Experiments" cell as needed

`%autoreload 2` is enabled in the Imports cell, so any edit to `src/` or `experiments/runner.py` is picked up automatically without restarting the kernel.

**Caveats of autoreload:** changes to `Enum`, `@dataclass(frozen=True)`, or class hierarchies are not reliably hot-patched. If you edit those, restart the kernel — the pickle cache will keep Load Data fast.

In [18]:
## Imports (run once)

# Auto-reload edited .py files before each cell execution.
# See top markdown for caveats (Enum / frozen dataclass changes still need a kernel restart).
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import multiprocessing as mp
import sys
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

_ROOT = Path(".").resolve().parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.config import DEFAULT_GPU_FLOPS, DEFAULT_PCIE_BANDWIDTH, DEFAULT_TOKENIZER_NAME, RESULTS_DIR
from src.model_config import DEFAULT_MODEL, ModelConfig
from experiments.runner import (
    capacity_from_spec,
    effective_page_size,
    persist_result_row,
    prepare_requests,
    run_simulation,
    strategy_from_name,
)
from viz.tree_logger import TreeLogger, _log_filename

print(f"Project root: {_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Default model: {DEFAULT_MODEL.name}  (available: {', '.join(ModelConfig.list_models())})")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Project root: /data/howarli/dev/LLM-prefix-caching-simulator
Results dir:  /data/howarli/dev/LLM-prefix-caching-simulator/results
Default model: qwen3.5-27b  (available: jamba-1.5-mini, llama-3.1-70b, llama-3.1-8b, qwen3-0.6b, qwen3.5-27b, transformer-28l-4096d)


## Configuration

Edit these and re-run. The "Load Data" cell uses `DATASETS`, `ORDERING`, `TOKENIZER`, `SEED`, `MAX_REQUESTS`, `TOKENIZE_WORKERS`.

In [19]:
# ── Data loading config ─────────────────────────────────────────────
DATASETS        = ["oasst1", "swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]
ORDERING        = "timestamp"
TOKENIZER       = DEFAULT_TOKENIZER_NAME
SEED            = 0
MAX_REQUESTS    = 5000
TOKENIZE_WORKERS = 90

# ── Model config ───────────────────────────────────────────────────
MODEL_NAME      = DEFAULT_MODEL.name  # see ModelConfig.list_models()

# ── Hardware throughput (used by saved-time metrics) ──────────────
GPU_FLOPS       = DEFAULT_GPU_FLOPS       # FLOP/sec; H100 BF16 dense, 20% effective values
PCIE_BANDWIDTH  = DEFAULT_PCIE_BANDWIDTH  # bytes/sec; PCIe Gen5 x16

# ── DRAM tier (two-tier cache) ─────────────────────────────────────
# Two-tier mode is enabled when DRAM_STRATEGY is non-empty AND DRAM_CAPACITY > 0.
# Per-experiment overrides via "dram_strategy" / "dram_capacity".
DRAM_STRATEGY   = None           # e.g. "marconi3_ev1_mn0" — None disables DRAM
DRAM_CAPACITY   = "0"            # GB (must be finite); "0" disables DRAM

# ── Logging ─────────────────────────────────────────────────────────
ENABLE_LOG      = False
LOG_DIR         = "viz/logs"

# ── Parallelism ─────────────────────────────────────────────────────
MAX_WORKERS     = 100

## Load Data

Tokenizes all datasets and caches the results in the in-memory `PREPARED` dict **and** in a disk pickle at `data/prepared/*.pkl` (handled inside `prepare_requests`). First run is slow (HF + tokenization); every subsequent run — even after restarting the kernel — is near-instant because `prepare_requests` short-circuits to the pickle.

Re-run this cell when you change `DATASETS`, `ORDERING`, `MAX_REQUESTS`, etc. To force a regeneration, delete the matching `data/prepared/*.pkl` file.

In [20]:
# PREPARED: dict mapping (dataset, ordering, tokenizer, max_requests, seed) -> List[TokenizedRequest]
PREPARED: Dict[tuple, list] = {}

for ds in DATASETS:
    prep_key = (ds, ORDERING, TOKENIZER, MAX_REQUESTS, SEED)
    if prep_key in PREPARED:
        print(f"  [skip] {ds}/{ORDERING} already loaded ({len(PREPARED[prep_key])} requests)")
        continue
    t0 = time.perf_counter()
    print(f"  Preparing: {ds}/{ORDERING} (max_requests={MAX_REQUESTS}) ...", end=" ", flush=True)
    reqs = prepare_requests(
        ds, ORDERING, TOKENIZER,
        seed=SEED,
        tokenize_workers=TOKENIZE_WORKERS,
        max_requests=MAX_REQUESTS,
    )
    elapsed = time.perf_counter() - t0
    PREPARED[prep_key] = reqs
    print(f"{len(reqs)} requests ready ({elapsed:.1f}s)")

print(f"\nTotal prep keys loaded: {len(PREPARED)}")

  Preparing: oasst1/timestamp (max_requests=5000) ... 5000 requests ready (1.1s)
  Preparing: swesmith/timestamp (max_requests=5000) ... 5000 requests ready (2.1s)
  Preparing: loogle/timestamp (max_requests=5000) ... 1951 requests ready (2.3s)
  Preparing: narrativeqa/timestamp (max_requests=5000) ... 1494 requests ready (4.6s)
  Preparing: sharegpt_90k_raw/timestamp (max_requests=5000) ... 5000 requests ready (5.7s)

Total prep keys loaded: 5


## Run Experiments

Edit the `EXPERIMENTS` list below and re-run. Thanks to `%autoreload 2`, edits to `src/strategies/*.py`, `src/cache_simulator.py`, etc. are picked up automatically — no need to restart the kernel or reload data.

In [25]:
# ── Define experiments ──────────────────────────────────────────────
# Edit this list freely and re-run this cell + the "Run all experiments" cell below.
# Edits to src/ .py files are auto-reloaded.

EXPERIMENTS: list[dict] = []


for ds in ["oasst1", "swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
# for ds in ["swesmith"]:
    # for page_size in [1, 32, 256, 1024]:
    for page_size in [32]:
        for capacity in [10, 20, 40, 80, 120, "inf"]:
        # for capacity in [20]:
            # continue
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="oracle_greedy", capacity=capacity))
            EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="oracle_ilp", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev0_nt_a0.3", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev1_nt_a0.3", capacity=capacity))
            # # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev2_nt", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev2_nt_a0.3", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev3_nt_a0.3", capacity=capacity))
            # # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="marconi3_ev1_nt_ef", capacity=capacity))
            # # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="branch_nt_ef", capacity=capacity))
            # EXPERIMENTS.append(dict(page_size=page_size, dataset=ds, strategy="branch_nt", capacity=capacity))


# Two-tier examples: HBM=branch_nt at small capacity, DRAM=marconi3_ev1_nt at large capacity.
for ds in ["oasst1", "swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
    for page_size in [32]:
        for hbm_cap in [10, 20]:
            for dram_cap in [64, 128, 256]:
                continue
                EXPERIMENTS.append(dict(
                    page_size=page_size, dataset=ds,
                    strategy="branch_nt", capacity=hbm_cap,
                    dram_strategy="marconi3_ev1_nt", dram_capacity=dram_cap,
                ))
                # EXPERIMENTS.append(dict(
                #     page_size=page_size, dataset=ds,
                #     strategy="marconi3_ev1_nt", capacity=hbm_cap,
                #     dram_strategy="marconi3_ev1_nt", dram_capacity=dram_cap,
                # ))

# cap_rate = 3  # DRAM capacity is this times HBM capacity
# for ds in ["swesmith", "loogle", "narrativeqa", "sharegpt_90k_raw"]:
#     for page_size in [32]:
#         for tot_cap in [10, 20, 40, 80, 120]:
#             hbm_cap = tot_cap / (1 + cap_rate)
#             dram_cap = tot_cap - hbm_cap
#             EXPERIMENTS.append(dict(
#                 page_size=page_size, dataset=ds,
#                 strategy="branch_nt", capacity=hbm_cap,
#                 dram_strategy="marconi3_ev1_nt", dram_capacity=dram_cap,
#             ))

print(f"Total experiments: {len(EXPERIMENTS)}")

Total experiments: 30


In [26]:
# ── Run all experiments ─────────────────────────────────────────────

def _build_strategy_name(name, cfg):
    if not name:
        return None
    n = name.lower()
    if n in ("marconi", "marconi2") and cfg.get("marconi_alpha") is not None:
        return f"{name}_{cfg['marconi_alpha']}"
    if n == "crf_decoupling" and cfg.get("crf_lambda_decay") is not None:
        return f"{name}_{cfg['crf_lambda_decay']}"
    return name

def _merge_defaults(cfg: dict) -> dict:
    return {
        "dataset":                cfg.get("dataset", DATASETS[0]),
        "page_size":              cfg.get("page_size", 32),
        "ordering":               cfg.get("ordering", ORDERING),
        "strategy":               cfg.get("strategy", "lru"),
        "capacity":               cfg.get("capacity", "160"),
        "model_name":             cfg.get("model_name", MODEL_NAME),
        "tokenizer":              cfg.get("tokenizer", TOKENIZER),
        "seed":                   cfg.get("seed", SEED),
        "max_requests":           cfg.get("max_requests", MAX_REQUESTS),
        "marconi_alpha":          cfg.get("marconi_alpha"),
        "crf_lambda_decay":       cfg.get("crf_lambda_decay"),
        "crf_c_attn":             cfg.get("crf_c_attn"),
        "crf_c_ssm":              cfg.get("crf_c_ssm"),
        # DRAM tier
        "dram_strategy":          cfg.get("dram_strategy", DRAM_STRATEGY),
        "dram_capacity":          cfg.get("dram_capacity", DRAM_CAPACITY),
        # Hardware throughput
        "gpu_flops":              cfg.get("gpu_flops", GPU_FLOPS),
        "pcie_bandwidth":         cfg.get("pcie_bandwidth", PCIE_BANDWIDTH),
    }

def _dram_enabled(cfg: dict) -> bool:
    return bool(cfg.get("dram_strategy")) and str(cfg.get("dram_capacity", "0")).strip() not in ("", "0")

def _label(cfg: dict) -> str:
    base = f"{cfg['dataset']} ps={cfg['page_size']} {cfg['ordering']} {cfg['strategy']} cap={cfg['capacity']}"
    if _dram_enabled(cfg):
        base += f" dram:{cfg['dram_strategy']}/{cfg['dram_capacity']}"
    return base

def _sim_worker(args):
    """Run one simulation inside a pool worker (fork inherits PREPARED)."""
    prep_key, page_size, sim_spec, cfg = args
    reqs = PREPARED[prep_key]
    model = ModelConfig.from_name(cfg["model_name"])

    strategy = strategy_from_name(
        sim_spec["strategy_name"],
        model=model,
        gpu_flops=cfg["gpu_flops"],
        pcie_bandwidth=cfg["pcie_bandwidth"],
        requests=reqs,
        page_size=page_size,
    )
    dram_strategy = None
    if sim_spec.get("dram_strategy_name"):
        dram_strategy = strategy_from_name(
            sim_spec["dram_strategy_name"],
            model=model,
            gpu_flops=cfg["gpu_flops"],
            pcie_bandwidth=cfg["pcie_bandwidth"],
        )

    logger = None
    if ENABLE_LOG:
        log_dir = Path(LOG_DIR)
        log_dir.mkdir(parents=True, exist_ok=True)
        fname = _log_filename(
            dataset=cfg["dataset"], strategy=sim_spec["strategy_name"],
            page_size=page_size, capacity_spec=str(cfg["capacity"]),
            ordering=cfg["ordering"], model_name=model.name,
            dram_strategy=sim_spec.get("dram_strategy_name"),
            dram_capacity_spec=str(cfg.get("dram_capacity", "0")),
        )
        logger = TreeLogger(log_dir / fname)

    try:
        metrics = run_simulation(
            reqs, page_size, strategy, sim_spec["cap"],
            dram_strategy=dram_strategy,
            dram_capacity_tokens=sim_spec.get("dram_cap", 0),
            logger=logger,
            model=model,
            gpu_flops=cfg["gpu_flops"],
            pcie_bandwidth=cfg["pcie_bandwidth"],
        )
    finally:
        if logger is not None:
            logger.close()

    return {"cfg": cfg, "metrics": metrics.to_dict()}


# Build simulation args
merged = [_merge_defaults(cfg) for cfg in EXPERIMENTS]
sim_args = []
for cfg in merged:
    ds = cfg["dataset"]
    model = ModelConfig.from_name(cfg["model_name"])
    prep_key = (ds, cfg["ordering"], cfg["tokenizer"], cfg["max_requests"], cfg["seed"])
    if prep_key not in PREPARED:
        print(f"  [WARNING] Data not loaded for {prep_key}, skipping")
        continue
    page_size = effective_page_size(ds, int(cfg["page_size"]))
    cap = capacity_from_spec(str(cfg["capacity"]), model=model)
    sim_spec = {
        "strategy_name": _build_strategy_name(cfg["strategy"], cfg),
        "cap": cap,
    }
    if _dram_enabled(cfg):
        # capacity_from_spec returns None for "inf" → unlimited DRAM.
        dram_cap = capacity_from_spec(str(cfg["dram_capacity"]), model=model)
        sim_spec["dram_strategy_name"] = _build_strategy_name(cfg["dram_strategy"], cfg)
        sim_spec["dram_cap"] = dram_cap
    sim_args.append((prep_key, page_size, sim_spec, cfg))

num_workers = min(MAX_WORKERS, len(sim_args))
print(f"Running {len(sim_args)} simulation(s) (model={MODEL_NAME}, num_workers={num_workers}) ...")
t_start = time.perf_counter()

ctx = mp.get_context("fork")
done = 0

with ctx.Pool(processes=num_workers) as pool:
    for result in pool.imap_unordered(_sim_worker, sim_args):
        cfg = result["cfg"]
        metrics_dict = result["metrics"]

        dataset = cfg["dataset"]
        out_csv = RESULTS_DIR / f"results_{dataset}.csv"
        out_json_dir = RESULTS_DIR / f"json_{dataset}"

        dram_enabled = _dram_enabled(cfg)
        row = {
            "dataset": dataset,
            "page_size": effective_page_size(dataset, int(cfg["page_size"])),
            "ordering": cfg["ordering"],
            "strategy": _build_strategy_name(cfg["strategy"], cfg),
            "capacity_spec": str(cfg["capacity"]),
            "dram_strategy": _build_strategy_name(cfg["dram_strategy"], cfg) or "" if dram_enabled else "",
            "dram_capacity_spec": str(cfg["dram_capacity"]) if dram_enabled else "",
            "tokenizer": cfg["tokenizer"],
            "model_name": cfg["model_name"],
            "gpu_flops": cfg["gpu_flops"],
            "pcie_bandwidth": cfg["pcie_bandwidth"],
            "metrics": metrics_dict,
        }
        persist_result_row(out_csv, out_json_dir, row)

        done += 1
        hbm_hr = metrics_dict.get("hbm_token_hit_rate", 0)
        dram_hr = metrics_dict.get("dram_token_hit_rate", 0)
        total_hr = hbm_hr + dram_hr
        flop_sr = metrics_dict.get("flop_save_rate")
        flop_str = f"  flop_save={flop_sr:.4f}" if flop_sr is not None else ""
        saved_t = metrics_dict.get("total_saved_time")
        saved_str = f"  saved_time={saved_t:.2f}s" if saved_t is not None else ""
        if dram_enabled:
            tier_str = f"  hbm_hr={hbm_hr:.4f}  dram_hr={dram_hr:.4f}"
        else:
            tier_str = ""
        print(f"  [{done}/{len(sim_args)}] {_label(cfg)}  token_hr={total_hr:.4f}{tier_str}{flop_str}{saved_str}", flush=True)

elapsed = time.perf_counter() - t_start
print(f"\nAll {len(sim_args)} experiments completed in {elapsed:.1f}s.")

Running 30 simulation(s) (model=qwen3.5-27b, num_workers=30) ...


ValueError: Unknown strategy 'oracle_ilp'